* denna notebook [388_SSOT_geojson.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/388_SSOT_geojson.ipynb)
* Issue [#388](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/388)

* Version 2.0 innehålla mer metadata
  * OSM relation
  * Wikdata Wnummer
  * SAT nummer
  * time
  * Datum
 
  * "OSM_REL": 19012436,
  * "QID": "Q133374147",
  * "Labelsv": "SAT Arholma",
  * "Labelen": "SAT Arholma",
  * "website_sv": "https://stockholmarchipelagotrail.com/sv/section/etapp-arholma/",
  * "website_en": "https://stockholmarchipelagotrail.com/section/sat-arholma/",
  * "P1401": "mailto:ProblemsOnSAT@gmail.com",
  * "P968": "mailto:GeneralinfoSAT@gmail.com",
  * "P373": "SAT Arholma"

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-06-07 21:15


In [2]:
import requests
import pandas as pd

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

query = """
SELECT DISTINCT ?P973 WHERE {
  ?item wdt:P361 wd:Q131318799 .

  ?item wdt:P973 ?P973 .

  FILTER(
    STRSTARTS(
      STR(?P973),
      "https://map.stockholmarchipelagotrail.com/data/geojson/islands/"
    )
  )
} order by ?item
"""

r = requests.get(
    SPARQL_ENDPOINT,
    params={"query": query, "format": "json"},
    headers={"User-Agent": "SAT-GeoJSON-Merge/1.0"}
)

r.raise_for_status()

urls = [
    b["P973"]["value"]
    for b in r.json()["results"]["bindings"]
]

print(f"{len(urls)} GeoJSON-filer hittades")
pd.DataFrame({"url": urls})

import requests
import json

all_features = []

for url in urls:
    print(url)

    data = requests.get(url, timeout=30).json()

    if data["type"] == "FeatureCollection":
        all_features.extend(data["features"])

    elif data["type"] == "Feature":
        all_features.append(data)

    else:
        print(f"Okänd GeoJSON-typ: {url}") 
from datetime import datetime, UTC

sat_full = {
    "type": "FeatureCollection",
    "name": "SAT_full",
    "generated": datetime.now(UTC).isoformat(),
    "source": "Wikidata P973",
    "feature_count": len(all_features),
    "features": all_features,
}

with open("SAT_full.geojson", "w", encoding="utf-8") as f:
    json.dump(sat_full, f, ensure_ascii=False)

print(f"Skapade SAT_full.geojson med {len(all_features)} features")

20 GeoJSON-filer hittades
https://map.stockholmarchipelagotrail.com/data/geojson/islands/arholma.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/lido.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/landsort.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/grinda.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/namdo.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/nattaro.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/rano.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/alo.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/uto.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/orno.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/fjardlang.geojson
https://map.stockholmarchipelagotrail.com/data/geojson/islands/svartso.geojson
https://map.stockholmarchipelagotrail.com/d

In [3]:
end_time = time.time()
duration = end_time - start_time

print(f"Finished in {duration:.2f} seconds.")


Finished in 2.38 seconds.
